In [1]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import requests
from dotenv import load_dotenv
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer

c:\Users\Himanshu\Desktop\CampusAI\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Reading documents from the file system
directory_path = "./Web_Scrapper"
loader = DirectoryLoader(
    path=directory_path,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}, # Add this line to specify encoding
    show_progress=True
)
docs = loader.load()
print(len(docs), "documents loaded.")

100%|██████████| 1/1 [00:00<00:00, 33.48it/s]

1 documents loaded.


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=250)
chunk = text_splitter.split_documents(docs)
print(len(chunk), "chunks created.")

1902 chunks created.


In [4]:
type(chunk[2])

langchain_core.documents.base.Document

In [5]:
chunk[2]

Document(metadata={'source': 'Web_Scrapper\\gehu-homepage.txt'}, page_content='All three campuses strive to create a nurturing environment that helps students grow and thrive in all aspects of their lives. Our campuses are home to a diverse student population who come from all over the world, contributing to a vibrant and inclusive learning community.\nAt GEHU it is our commitment to provide high-quality education and research opportunities that contribute to knowledge creation and innovation, not just in our region, but on an international scale. Our faculty members are experts in their respective fields and are dedicated to inspiring and mentoring the next generation of leaders.\nGEHU Dehradun, Bhimtal, and Haldwani campuses provide a range of co-curricular activities, cultural events, and social initiatives that help students to develop their skills, passions, and interests outside of the classroom.')

In [6]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
text = [doc.page_content for doc in chunk]
embeddings = model.encode(
    text,
    batch_size=32,
    show_progress_bar=True
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 203.61it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 60/60 [01:20<00:00,  1.35s/it]


In [7]:
print(len(embeddings), "embeddings created.")
print("Dimension of each embedding:", len(embeddings[0]))
print("Sample embedding:", embeddings[0][:5])
print(len(chunk), "chunks and", len(embeddings), "embeddings should match.")

1902 embeddings created.
Dimension of each embedding: 384
Sample embedding: [ 0.01955767  0.07343878 -0.01102463 -0.08509269 -0.00771436]
1902 chunks and 1902 embeddings should match.


In [10]:
client = MongoClient(os.getenv("MONGO_URI"))

db = client["campus_ai"]
collection = db["academic_vector"]
vectors = []

for i, embedding in enumerate(embeddings):
    vectors.append({
        "id": f"chunk-{i}",
        "values": embedding.tolist(),
        "metadata": {
            "text": chunk[i].page_content,
            "source": chunk[i].metadata.get("source"),
            "page": chunk[i].metadata.get("page")
        }
    })
collection.insert_many(vectors)

InsertManyResult([ObjectId('69a095bedaf8f84b9381ac5b'), ObjectId('69a095bedaf8f84b9381ac5c'), ObjectId('69a095bedaf8f84b9381ac5d'), ObjectId('69a095bedaf8f84b9381ac5e'), ObjectId('69a095bedaf8f84b9381ac5f'), ObjectId('69a095bedaf8f84b9381ac60'), ObjectId('69a095bedaf8f84b9381ac61'), ObjectId('69a095bedaf8f84b9381ac62'), ObjectId('69a095bedaf8f84b9381ac63'), ObjectId('69a095bedaf8f84b9381ac64'), ObjectId('69a095bedaf8f84b9381ac65'), ObjectId('69a095bedaf8f84b9381ac66'), ObjectId('69a095bedaf8f84b9381ac67'), ObjectId('69a095bedaf8f84b9381ac68'), ObjectId('69a095bedaf8f84b9381ac69'), ObjectId('69a095bedaf8f84b9381ac6a'), ObjectId('69a095bedaf8f84b9381ac6b'), ObjectId('69a095bedaf8f84b9381ac6c'), ObjectId('69a095bedaf8f84b9381ac6d'), ObjectId('69a095bedaf8f84b9381ac6e'), ObjectId('69a095bedaf8f84b9381ac6f'), ObjectId('69a095bedaf8f84b9381ac70'), ObjectId('69a095bedaf8f84b9381ac71'), ObjectId('69a095bedaf8f84b9381ac72'), ObjectId('69a095bedaf8f84b9381ac73'), ObjectId('69a095bedaf8f84b9381ac

In [27]:
query_vector = model.encode(["tell about placements"])[0].tolist()

pipeline = [
    {
        "$vectorSearch": {
            "index": "academic_index",
            "path": "values",
            "queryVector": query_vector,
            "numCandidates": 100,
            "limit": 5
        }
    },
    {
            "$project": {
                "_id": 0,
                "id": 1,
                "metadata": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
]

results = list(collection.aggregate(pipeline))
# print(results)
for doc in results:
    print(doc["metadata"]["text"])
    print(doc["score"])

- Department of Electronics and Communication Engineering
- Top Placements are a result of synchronized performing of a lot many aspects of the higher education ecosystem, such as student performance and skillsets, the academic rigor instilled in the course structure and content delivery, specific placement preparation for specialised roles and a specialized and professional Placement Team bridging the opportunities with the right candidates. Graphic Era’s Placement Ecosystem excels in all of these aspects and this is the recipe for our success. The Academic Rigor and the student performance and skillsets, go hand in hand and this is found in the very basic structure of the University.
0.8325449228286743
- Department of Electronics and Communication Engineering
- Top Placements are a result of synchronized performing of a lot many aspects of the higher education ecosystem, such as student performance and skillsets, the academic rigor instilled in the course structure and content delive